<a href="https://colab.research.google.com/github/YOUR_USERNAME/time/blob/main/tta_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<p><small>If the link 404s, push this repo to your GitHub and replace YOUR_USERNAME, or run this notebook after uploading it to Colab.</small></p>

# Temporal Truth Architecture (TTA) Benchmark
## With Semantic Retrieval for Google Colab (H100 GPU)

This notebook runs the TTA benchmark with semantic retrieval using sentence embeddings.
Optimized for H100 GPU on Google Colab.

In [ ]:
# @title ## Setup: Install Dependencies
!pip install sentence-transformers scikit-learn -q
print("✓ Dependencies installed")

In [ ]:
# @title ## Setup: Clone Repository
# Set your repo URL (use your GitHub username/fork if the default doesn't exist)
REPO_URL = "https://github.com/YOUR_USERNAME/time.git"  # e.g. https://github.com/youruser/time.git
REPO_DIR = "/content/time"

import os
import shutil
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
exit_code = os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
if exit_code != 0:
    print("Clone failed (404 = repo not found or private). Options:")
    print("  1. Push this repo to your GitHub and set REPO_URL above to your repo.")
    print("  2. Upload repo as zip: zip the repo locally, then in a new cell run:")
    print("     from google.colab import files; files.upload()  # choose your time.zip")
    print("     !unzip -q -o time.zip -d /content && (mv /content/time-main /content/time 2>/dev/null || true) && %cd /content/time")
    raise SystemExit(1)
%cd /content/time
print("✓ Repository cloned")

In [ ]:
# @title ## Setup: Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Run Benchmark

In [ ]:
# @title ### Option 1: Run All Systems (A, B, C, D, E, E_hybrid)
# This runs the full benchmark including semantic retrieval systems
%cd /content/time
!python scripts/run_benchmark.py --semantic

In [ ]:
# @title ### Option 2: Run Specific Systems
# Customize which systems to run
%cd /content/time
!python scripts/run_benchmark.py --systems A,B,C,D,E,E_hybrid --out results/colab_results.csv

In [ ]:
# @title ### Option 3: Run on Different Benchmark Versions
# v1: Easy, v2: Adversarial, v3: Hard, v4: Extreme
%cd /content/time

# Run on v3 (Hard) benchmark
!python scripts/run_benchmark.py --facts benchmarks/temporalbench_v3_facts.jsonl \
  --questions benchmarks/temporalbench_v3_questions.jsonl \
  --events benchmarks/temporalbench_v3_events.jsonl \
  --systems A,B,C,D,E,E_hybrid \
  --out results/v3_with_semantic.csv

## View Results

In [ ]:
import pandas as pd

# Load and display results
results = pd.read_csv('results/main_table_with_semantic.csv')
print("=== Benchmark Results ===\n")
print(results.to_string(index=False))

In [ ]:
# Compare systems side by side
import matplotlib.pyplot as plt

systems = results['system']
trs_scores = results['TRS']
accuracy = results['TemporalAccuracy'] * 100
staleness = results['StalenessErrorRate'] * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# TRS Score
axes[0].bar(systems, trs_scores, color=['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4', '#ffeaa7', '#dfe6e9'])
axes[0].set_ylabel('TRS Score')
axes[0].set_title('Temporal Reliability Score')
axes[0].set_ylim(0, 1.1)

# Temporal Accuracy
axes[1].bar(systems, accuracy, color=['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4', '#ffeaa7', '#dfe6e9'])
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Temporal Accuracy')

# Staleness Error Rate
axes[2].bar(systems, staleness, color=['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4', '#ffeaa7', '#dfe6e9'])
axes[2].set_ylabel('Error Rate (%)')
axes[2].set_title('Staleness Error Rate')

plt.tight_layout()
plt.savefig('results/benchmark_comparison.png', dpi=150)
plt.show()
print("✓ Chart saved to results/benchmark_comparison.png")

## System Descriptions

| System | Description |
|--------|-------------|
| **A** (Plain RAG) | Naive retrieval - picks most recent fact by timestamp |
| **B** (Temporal Rerank) | Temporal validity + recency scoring, no decay |
| **C** (Time Constraint) | Explicit temporal validity filtering |
| **D** (TTA Full) | Truth intervals + domain-adaptive decay + reranking |
| **E** (Semantic) | Dense embeddings for semantic matching + temporal filtering |
| **E_hybrid** (Hybrid) | Exact match first, then semantic fallback |

### Key Insights
- Systems A-D are rule-based retrieval
- Systems E/E_hybrid use neural embeddings (sentence-transformers)
- H100 GPU accelerates embedding computation significantly
- Semantic retrieval helps when questions don't exactly match fact subjects